In [2]:
import numpy as np
import pandas as pd
import joblib
import matplotlib.pyplot as plt

from sklearn.metrics import confusion_matrix

In [3]:
X_test_zero_day = joblib.load("../data/X_test_zero_day.pkl")
y_test_zero_day = joblib.load("../data/y_test_zero_day.pkl")
X_train_scaled = joblib.load("../data/X_train_zero_day_scaled.pkl")

X_test_seen_scaled = joblib.load("../data/X_test_seen_scaled.pkl")

X_test_zero_day_scaled = joblib.load("../data/X_test_zero_day_scaled.pkl")
print("Zero-Day Test Shape:", X_test_zero_day.shape)
print("Zero-Day Labels Shape:", y_test_zero_day.shape)

Zero-Day Test Shape: (3750, 122)
Zero-Day Labels Shape: (3750,)


In [4]:
test_path = "../data/KDDTest+.txt"

test = pd.read_csv(test_path, header=None)

columns = [
    "duration",
    "protocol_type",
    "service",
    "flag",
    "src_bytes",
    "dst_bytes",
    "land",
    "wrong_fragment",
    "urgent",
    "hot",
    "num_failed_logins",
    "logged_in",
    "num_compromised",
    "root_shell",
    "su_attempted",
    "num_root",
    "num_file_creations",
    "num_shells",
    "num_access_files",
    "num_outbound_cmds",
    "is_host_login",
    "is_guest_login",
    "count",
    "srv_count",
    "serror_rate",
    "srv_serror_rate",
    "rerror_rate",
    "srv_rerror_rate",
    "same_srv_rate",
    "diff_srv_rate",
    "srv_diff_host_rate",
    "dst_host_count",
    "dst_host_srv_count",
    "dst_host_same_srv_rate",
    "dst_host_diff_srv_rate",
    "dst_host_same_src_port_rate",
    "dst_host_srv_diff_host_rate",
    "dst_host_serror_rate",
    "dst_host_srv_serror_rate",
    "dst_host_rerror_rate",
    "dst_host_srv_rerror_rate",
    "label",
    "difficulty"
]

test.columns = columns

train = pd.read_csv("../data/KDDTrain+.txt", header=None)
train.columns = columns

train_attacks = set(
    train.loc[train["label"] != "normal", "label"].unique()
)

test_attacks = set(
    test.loc[test["label"] != "normal", "label"].unique()
)

unseen_attacks = sorted(test_attacks - train_attacks)

test_zero_day = test[
    test["label"].isin(unseen_attacks)
].copy()

print("Zero-Day attack types:")
print(test_zero_day["label"].value_counts().sort_index())

print("\nTotal Zero-Day samples:", len(test_zero_day))

Zero-Day attack types:
label
apache2          737
httptunnel       133
mailbomb         293
mscan            996
named             17
processtable     685
ps                15
saint            319
sendmail          14
snmpgetattack    178
snmpguess        331
sqlattack          2
udpstorm           2
worm               2
xlock              9
xsnoop             4
xterm             13
Name: count, dtype: int64

Total Zero-Day samples: 3750


In [5]:
supervised_models = {
    "Random Forest": joblib.load("../models/random_forest_optimized.pkl"),
    "Decision Tree": joblib.load("../models/decision_tree_regularized.pkl"),
    "KNN": joblib.load("../models/knn_baseline.pkl"),
    "Linear SVM": joblib.load("../models/linear_svm_baseline.pkl"),
    "Logistic Regression": joblib.load("../models/logistic_regression_baseline.pkl")
}

unsupervised_models = {
    "Isolation Forest": joblib.load("../models/isolation_forest.pkl"),
    "One-Class SVM": joblib.load("../models/one_class_svm.pkl"),
    "LOF": joblib.load("../models/local_outlier_factor.pkl")
}

supervised_thresholds = joblib.load(
    "../models/supervised_thresholds.pkl"
)

unsupervised_thresholds = joblib.load(
    "../models/unsupervised_thresholds.pkl"
)

print("Supervised Models:")
print(list(supervised_models.keys()))

print("\nUnsupervised Models:")
print(list(unsupervised_models.keys()))

print("\nSupervised Thresholds:")
print(supervised_thresholds)

print("\nUnsupervised Thresholds:")
print(unsupervised_thresholds)

Supervised Models:
['Random Forest', 'Decision Tree', 'KNN', 'Linear SVM', 'Logistic Regression']

Unsupervised Models:
['Isolation Forest', 'One-Class SVM', 'LOF']

Supervised Thresholds:
{'random_forest': np.float64(0.19), 'decision_tree': np.float64(0.01)}

Unsupervised Thresholds:
{'isolation_forest': np.float64(-0.007978589803472214), 'one_class_svm': np.float64(-1.8493333982406948e-06), 'local_outlier_factor': np.float64(0.11672763183658934)}


In [6]:
predictions = {}

# Supervised
rf = supervised_models["Random Forest"]
dt = supervised_models["Decision Tree"]
knn = supervised_models["KNN"]
svm = supervised_models["Linear SVM"]
lr = supervised_models["Logistic Regression"]

rf_scores = rf.predict_proba(X_test_zero_day)[:, 1]
dt_scores = dt.predict_proba(X_test_zero_day)[:, 1]

predictions["Random Forest"] = (
    rf_scores >= supervised_thresholds["random_forest"]
).astype(int)

predictions["Decision Tree"] = (
    dt_scores >= supervised_thresholds["decision_tree"]
).astype(int)

predictions["KNN"] = knn.predict(X_test_zero_day_scaled)

predictions["Linear SVM"] = svm.predict(X_test_zero_day_scaled)

predictions["Logistic Regression"] = lr.predict(X_test_zero_day_scaled)


# Unsupervised
iso = unsupervised_models["Isolation Forest"]
ocsvm = unsupervised_models["One-Class SVM"]
lof = unsupervised_models["LOF"]

iso_scores = -iso.decision_function(X_test_zero_day)
ocsvm_scores = -ocsvm.decision_function(X_test_zero_day)
lof_scores = -lof.decision_function(X_test_zero_day)

predictions["Isolation Forest"] = (
    iso_scores >= unsupervised_thresholds["isolation_forest"]
).astype(int)

predictions["One-Class SVM"] = (
    ocsvm_scores >= unsupervised_thresholds["one_class_svm"]
).astype(int)

predictions["LOF"] = (
    lof_scores >= unsupervised_thresholds["local_outlier_factor"]
).astype(int)


print("Predictions generated successfully.\n")

for model_name, pred in predictions.items():
    print(
        f"{model_name}: "
        f"{pred.sum()} / {len(pred)} detected "
        f"({pred.mean() * 100:.2f}%)"
    )

Predictions generated successfully.

Random Forest: 2134 / 3750 detected (56.91%)
Decision Tree: 1876 / 3750 detected (50.03%)
KNN: 1632 / 3750 detected (43.52%)
Linear SVM: 1370 / 3750 detected (36.53%)
Logistic Regression: 1524 / 3750 detected (40.64%)
Isolation Forest: 2448 / 3750 detected (65.28%)
One-Class SVM: 1632 / 3750 detected (43.52%)
LOF: 1001 / 3750 detected (26.69%)


In [12]:
per_attack_results = []

for attack in sorted(test_zero_day["label"].unique()):

    attack_mask = test_zero_day["label"] == attack
    mask = attack_mask.to_numpy()

    for model_name, pred in predictions.items():

        detection_rate = pred[mask].mean() * 100

        per_attack_results.append({
            "Attack Type": attack,
            "Model": model_name,
            "Detection Rate (%)": detection_rate,
            "Samples": mask.sum()
        })

per_attack_df = pd.DataFrame(per_attack_results)

per_attack_df

,Attack Type,Model,Detection Rate (%),Samples
0,apache2,Random Forest,68.113976,737
1,apache2,Decision Tree,67.299864,737
2,apache2,KNN,70.827680,737
3,apache2,Linear SVM,99.050204,737
4,apache2,Logistic Regression,99.050204,737
...,...,...,...,...
131,xterm,Linear SVM,15.384615,13
132,xterm,Logistic Regression,30.769231,13
133,xterm,Isolation Forest,46.153846,13
134,xterm,One-Class SVM,15.384615,13


In [13]:
per_attack_pivot = per_attack_df.pivot(
    index="Attack Type",
    columns="Model",
    values="Detection Rate (%)"
).round(2)

per_attack_pivot

Model,Decision Tree,Isolation Forest,KNN,LOF,Linear SVM,Logistic Regression,One-Class SVM,Random Forest
Attack Type,,,,,,,,
apache2,67.30,95.93,70.83,65.54,99.05,99.05,94.98,68.11
httptunnel,84.21,84.21,12.03,10.53,1.50,1.50,66.92,82.71
mailbomb,3.41,0.00,0.00,0.00,0.00,0.00,0.00,1.71
mscan,88.25,95.08,76.91,11.85,32.63,47.79,15.56,60.84
named,52.94,23.53,23.53,41.18,5.88,5.88,47.06,41.18
processtable,2.04,53.87,0.58,4.53,0.44,0.29,68.47,34.31
ps,60.00,13.33,13.33,20.00,0.00,0.00,0.00,66.67
saint,99.69,94.04,95.92,90.60,95.30,95.61,63.01,100.00
sendmail,42.86,0.00,28.57,50.00,7.14,7.14,0.00,0.00
